# RL Graph Conjecture Counterexample Search

## Deep Cross-Entropy Method for Disproving Laplacian Spectral Radius Bounds

This notebook implements Wagner's (2021) Deep Cross-Entropy Method (CEM) in PyTorch to search for
counterexamples to conjectured upper bounds on the largest Laplacian eigenvalue of graphs.

### Background

**Wagner (2021)** showed that reinforcement learning can discover counterexamples to open graph theory
conjectures by framing graph construction as a sequential decision process. A neural network learns
to predict which edges to include, guided by the cross-entropy method.

**Ghebleh et al. (2024)** applied this approach systematically to 68 conjectured Laplacian spectral
radius bounds (Brankov et al.), disproving 30 of them. 38 bounds remain open.

### Method

- **Graph Construction MDP**: At each step, decide whether to add or skip an edge in the upper triangle
  of the adjacency matrix. After all N*(N-1)/2 decisions, evaluate the resulting graph.
- **State**: Binary adjacency vector (edges decided so far) + one-hot position marker (current edge).
- **Reward**: Violation degree of the conjectured bound. Positive reward = counterexample found.
- **Training**: Cross-entropy method selects elite sessions (top 7%) to train the neural network.

### References

- Wagner, A.Z. (2021). "Constructions in combinatorics via neural networks." *arXiv:2104.14516*
- Ghebleh, M. et al. (2024). "Refuting conjectures on Laplacian spectral radius bounds using ML."
- Brankov, V. et al. "Automated conjectures on upper bounds for the largest Laplacian eigenvalue."

In [ ]:
"""Cell 2: Imports and Configuration"""

import torch
import torch.nn as nn
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

# ── Hyperparameters ──────────────────────────────────────────────
N = 20                      # Number of vertices (overridden per run)
DECISIONS = N * (N - 1) // 2
OBSERVATION_SPACE = 2 * DECISIONS

N_SESSIONS = 1000           # Sessions per generation
PERCENTILE = 93             # Elite threshold (top 7%)
SUPER_PERCENTILE = 94       # Super-elite threshold (top 6%)
LEARNING_RATE = 0.0001      # SGD learning rate
MAX_ITERATIONS = 1000       # Default max iterations

DISCONNECTED_PENALTY = -1_000_000  # Reward for disconnected graphs

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device (CPU — GPU unnecessary for this small MLP)
DEVICE = torch.device('cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"NetworkX version: {nx.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Device: {DEVICE}")
print(f"Default graph size N={N}, decisions={DECISIONS}, state_dim={OBSERVATION_SPACE}")

In [ ]:
"""Cell 3: CEM Model — MLP (128 → 64 → 4 → 1)"""

class CEMModel(nn.Module):
    """
    Wagner's Deep Cross-Entropy Method neural network.
    
    Architecture: Linear(input_dim, 128) → ReLU → Linear(128, 64) → ReLU
                  → Linear(64, 4) → ReLU → Linear(4, 1) → Sigmoid
    
    Input: state vector of dimension 2 * N*(N-1)/2
           (adjacency upper triangle + one-hot position marker)
    Output: probability of adding the current edge (scalar in [0, 1])
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)


def create_model(n_vertices):
    """Create a new CEM model for graphs with n_vertices nodes."""
    decisions = n_vertices * (n_vertices - 1) // 2
    input_dim = 2 * decisions
    model = CEMModel(input_dim).to(DEVICE)
    return model


# Quick architecture check
_test_model = create_model(10)
_test_input = torch.zeros(1, 2 * 45)  # n=10 → 45 decisions → dim=90
_test_output = _test_model(_test_input)
print(f"Model test (n=10): input_dim=90, output={_test_output.item():.4f}")
n_params = sum(p.numel() for p in _test_model.parameters())
print(f"Total parameters: {n_params}")
del _test_model, _test_input, _test_output

In [ ]:
"""Cell 4: Graph Utility Functions"""

def edge_index_to_pair(idx, n):
    """Convert flat upper-triangle index to (i, j) vertex pair.

    Edge ordering: (0,1), (0,2), ..., (0,n-1), (1,2), ..., (n-2,n-1)
    """
    # Row i where the edge falls
    i = 0
    cumulative = 0
    for row in range(n - 1):
        row_edges = n - 1 - row
        if cumulative + row_edges > idx:
            i = row
            j = i + 1 + (idx - cumulative)
            return (i, j)
        cumulative += row_edges
    raise ValueError(f"Invalid edge index {idx} for n={n}")


def build_graph(state, n):
    """Build a NetworkX graph from the binary adjacency vector.

    Args:
        state: binary vector of length >= n*(n-1)/2 (first half of full state)
        n: number of vertices

    Returns:
        NetworkX Graph
    """
    G = nx.Graph()
    G.add_nodes_from(range(n))
    decisions = n * (n - 1) // 2
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if state[count] == 1:
                G.add_edge(i, j)
            count += 1
    return G


def compute_laplacian_spectral_radius(G):
    """Compute the largest eigenvalue of the Laplacian matrix L = D - A.

    Args:
        G: NetworkX graph (must have at least 1 edge)

    Returns:
        mu: largest Laplacian eigenvalue (float)
    """
    if G.number_of_edges() == 0:
        return 0.0
    L = nx.laplacian_matrix(G).toarray().astype(np.float64)
    eigenvalues = np.linalg.eigvalsh(L)
    return float(eigenvalues[-1])


def compute_dv_mv(G, n):
    """Compute degree (d_v) and average neighbor degree (m_v) for all vertices.

    Args:
        G: NetworkX graph
        n: number of vertices

    Returns:
        dv: array of degrees, shape (n,)
        mv: array of average neighbor degrees, shape (n,)
             (m_v = 0 for isolated vertices)
    """
    dv = np.zeros(n, dtype=np.float64)
    mv = np.zeros(n, dtype=np.float64)

    for v in range(n):
        d = G.degree(v)
        dv[v] = d
        if d > 0:
            neighbor_degree_sum = sum(G.degree(u) for u in G.neighbors(v))
            mv[v] = neighbor_degree_sum / d

    return dv, mv


# Precompute edge pairs lookup for efficiency
def precompute_edge_pairs(n):
    """Precompute the mapping from flat index to (i, j) pairs."""
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((i, j))
    return pairs


# ── Numpy bypass functions ───────────────────────────────────────
# These replace NetworkX in the CEM hot loop for ~5-10x speedup.
# verify_counterexample() still uses NetworkX for independent cross-verification.

def precompute_triu_indices(n):
    """Precompute upper triangle indices for vectorized adjacency matrix construction."""
    rows, cols = [], []
    for i in range(n):
        for j in range(i + 1, n):
            rows.append(i)
            cols.append(j)
    return np.array(rows, dtype=np.int32), np.array(cols, dtype=np.int32)


def compute_graph_properties_numpy(state, n, triu_rows, triu_cols):
    """All-numpy graph property computation (no NetworkX).

    Replaces build_graph + compute_laplacian_spectral_radius + compute_dv_mv
    in a single function for maximum performance.

    Args:
        state: binary adjacency vector, length >= n*(n-1)/2
        n: number of vertices
        triu_rows: precomputed row indices from precompute_triu_indices
        triu_cols: precomputed col indices from precompute_triu_indices

    Returns:
        tuple (mu, dv, mv, A) or None if graph is disconnected
        mu: largest Laplacian eigenvalue (float)
        dv: degree vector, shape (n,)
        mv: average neighbor degree vector, shape (n,)
        A: adjacency matrix, shape (n, n)
    """
    decisions = n * (n - 1) // 2

    # Vectorized adjacency matrix construction
    A = np.zeros((n, n), dtype=np.float64)
    mask = state[:decisions] == 1
    A[triu_rows[mask], triu_cols[mask]] = 1.0
    A[triu_cols[mask], triu_rows[mask]] = 1.0

    # Degree vector
    dv = A.sum(axis=1)

    # Quick disconnection check: isolated vertices
    if np.any(dv == 0):
        return None

    # DFS connectivity check (numpy-assisted)
    visited = np.zeros(n, dtype=bool)
    stack = [0]
    visited[0] = True
    count = 1
    while stack:
        v = stack.pop()
        for u in np.nonzero(A[v])[0]:
            if not visited[u]:
                visited[u] = True
                stack.append(int(u))
                count += 1
    if count < n:
        return None

    # Laplacian spectral radius via eigvalsh
    L = np.diag(dv) - A
    mu = float(np.linalg.eigvalsh(L)[-1])

    # Average neighbor degree via matrix multiply: (A @ dv)[v] / dv[v]
    mv = np.zeros(n, dtype=np.float64)
    nonzero = dv > 0
    mv[nonzero] = (A @ dv)[nonzero] / dv[nonzero]

    return mu, dv, mv, A


print("Graph utility functions defined (NetworkX + numpy bypass).")

In [ ]:
"""Cell 5: Unit Tests — Laplacian Spectral Radius"""

def run_unit_tests():
    """Test compute_laplacian_spectral_radius on known graphs.

    Test cases (regular):
    1. Complete graph K_5: mu = 5.0
    2. Star graph S_5 (1 center + 4 leaves): mu = 5.0
    3. Cycle graph C_6: mu = 4.0

    Test cases (non-regular):
    4. Path graph P_6: mu = 2 + sqrt(3) ≈ 3.732051
    5. Wheel graph W_6: mu = 6.0
    6. Barbell graph B(4,1): mu via eigvalsh reference
    """
    results = []
    tol = 1e-6

    # Test 1: Complete graph K_5
    G_k5 = nx.complete_graph(5)
    mu_k5 = compute_laplacian_spectral_radius(G_k5)
    expected_k5 = 5.0
    pass_k5 = abs(mu_k5 - expected_k5) < tol
    results.append(('K_5', expected_k5, mu_k5, pass_k5))

    # Test 2: Star graph S_5 (1 center + 4 leaves = 5 nodes)
    G_s5 = nx.star_graph(4)
    mu_s5 = compute_laplacian_spectral_radius(G_s5)
    expected_s5 = 5.0
    pass_s5 = abs(mu_s5 - expected_s5) < tol
    results.append(('S_5', expected_s5, mu_s5, pass_s5))

    # Test 3: Cycle graph C_6
    G_c6 = nx.cycle_graph(6)
    mu_c6 = compute_laplacian_spectral_radius(G_c6)
    expected_c6 = 4.0
    pass_c6 = abs(mu_c6 - expected_c6) < tol
    results.append(('C_6', expected_c6, mu_c6, pass_c6))

    # Test 4: Path graph P_6 (non-regular)
    # Laplacian eigenvalues of P_n: 2 - 2*cos(pi*k/n) for k=0..n-1
    # P_6 largest: 2 - 2*cos(5*pi/6) = 2 + sqrt(3)
    G_p6 = nx.path_graph(6)
    mu_p6 = compute_laplacian_spectral_radius(G_p6)
    expected_p6 = 2.0 + np.sqrt(3.0)
    pass_p6 = abs(mu_p6 - expected_p6) < tol
    results.append(('P_6 (non-reg)', expected_p6, mu_p6, pass_p6))

    # Test 5: Wheel graph W_6 (non-regular, 6 vertices)
    # W_n = K_1 + C_{n-1}. For W_6: center degree=5, rim degree=3.
    # Largest Laplacian eigenvalue of W_6 = 6.0
    G_w6 = nx.wheel_graph(6)
    mu_w6 = compute_laplacian_spectral_radius(G_w6)
    expected_w6 = 6.0
    pass_w6 = abs(mu_w6 - expected_w6) < tol
    results.append(('W_6 (non-reg)', expected_w6, mu_w6, pass_w6))

    # Test 6: Barbell graph B(4,1) (non-regular, 9 vertices)
    # Two K_4 cliques connected by a single bridge vertex
    G_barbell = nx.barbell_graph(4, 1)
    mu_barbell = compute_laplacian_spectral_radius(G_barbell)
    L_barbell = nx.laplacian_matrix(G_barbell).toarray().astype(np.float64)
    expected_barbell = float(np.linalg.eigvalsh(L_barbell)[-1])
    pass_barbell = abs(mu_barbell - expected_barbell) < tol
    results.append(('Barbell(4,1) (non-reg)', expected_barbell, mu_barbell, pass_barbell))

    # Print results
    print("=" * 65)
    print("UNIT TESTS: Laplacian Spectral Radius")
    print("=" * 65)
    all_pass = True
    for name, expected, actual, passed in results:
        status = "PASS" if passed else "FAIL"
        if not passed:
            all_pass = False
        print(f"  {name}: expected={expected:.6f}, got={actual:.6f} -> [{status}]")
    print("=" * 65)

    if all_pass:
        print("All unit tests PASSED.")
    else:
        print("WARNING: Some tests FAILED!")

    # Build-graph roundtrip test
    print("\nBuild-graph roundtrip test:")
    n_test = 5
    decisions_test = n_test * (n_test - 1) // 2
    state_k5 = np.ones(decisions_test, dtype=np.int32)
    G_rebuilt = build_graph(state_k5, n_test)
    assert G_rebuilt.number_of_edges() == 10, f"Expected 10 edges, got {G_rebuilt.number_of_edges()}"
    mu_rebuilt = compute_laplacian_spectral_radius(G_rebuilt)
    assert abs(mu_rebuilt - 5.0) < tol, f"Expected mu=5.0, got {mu_rebuilt}"
    print(f"  K_5 from state vector: edges={G_rebuilt.number_of_edges()}, mu={mu_rebuilt:.6f} -> [PASS]")

    dv, mv = compute_dv_mv(G_rebuilt, n_test)
    assert np.all(dv == 4), f"K_5 degrees should all be 4, got {dv}"
    assert np.all(mv == 4), f"K_5 avg neighbor degree should all be 4, got {mv}"
    print(f"  K_5 dv={dv}, mv={mv} -> [PASS]")

    dv_s, mv_s = compute_dv_mv(G_s5, 5)
    print(f"  S_5 dv={dv_s}, mv={mv_s}")
    assert dv_s[0] == 4, f"Star center degree should be 4, got {dv_s[0]}"
    assert np.all(dv_s[1:] == 1), f"Star leaf degrees should be 1"
    print(f"  S_5 dv/mv check -> [PASS]")

    # ── Numpy bypass cross-verification ──────────────────────────
    print("\n" + "=" * 65)
    print("CROSS-VERIFICATION: NetworkX vs Numpy bypass")
    print("=" * 65)

    numpy_pass = True

    test_graphs = {
        'K_5': (G_k5, 5),
        'S_5': (G_s5, 5),
        'C_6': (G_c6, 6),
        'P_6': (G_p6, 6),
        'W_6': (G_w6, 6),
        'Barbell(4,1)': (G_barbell, 9),
    }

    for name, (G, n_v) in test_graphs.items():
        decisions = n_v * (n_v - 1) // 2
        state = np.zeros(decisions, dtype=np.int32)
        idx = 0
        for i in range(n_v):
            for j in range(i + 1, n_v):
                if G.has_edge(i, j):
                    state[idx] = 1
                idx += 1

        triu_r, triu_c = precompute_triu_indices(n_v)
        result_np = compute_graph_properties_numpy(state, n_v, triu_r, triu_c)

        mu_nx = compute_laplacian_spectral_radius(G)
        dv_nx, mv_nx = compute_dv_mv(G, n_v)

        if result_np is None:
            print(f"  {name}: numpy returned None (disconnected) -> [FAIL]")
            numpy_pass = False
        else:
            mu_np, dv_np, mv_np, A_np = result_np
            mu_match = abs(mu_np - mu_nx) < 1e-10
            dv_match = np.allclose(dv_np, dv_nx, atol=1e-10)
            mv_match = np.allclose(mv_np, mv_nx, atol=1e-10)
            all_match = mu_match and dv_match and mv_match
            status = "PASS" if all_match else "FAIL"
            if not all_match:
                numpy_pass = False
            print(f"  {name}: mu_diff={abs(mu_np-mu_nx):.2e}, "
                  f"dv_match={dv_match}, mv_match={mv_match} -> [{status}]")

    # Random graph stress test: n=15, 50 random graphs
    print("\n  Random graph stress test (n=15, 50 graphs):")
    n_stress = 15
    n_tests = 50
    triu_r15, triu_c15 = precompute_triu_indices(n_stress)
    decisions_15 = n_stress * (n_stress - 1) // 2
    stress_pass = 0
    stress_fail = 0

    rng = np.random.RandomState(42)
    for t in range(n_tests):
        state_rand = rng.randint(0, 2, size=decisions_15).astype(np.int32)

        G_rand = build_graph(state_rand, n_stress)
        if not nx.is_connected(G_rand):
            result_np = compute_graph_properties_numpy(state_rand, n_stress, triu_r15, triu_c15)
            if result_np is None:
                stress_pass += 1
            else:
                stress_fail += 1
            continue

        mu_nx = compute_laplacian_spectral_radius(G_rand)
        dv_nx, mv_nx = compute_dv_mv(G_rand, n_stress)

        result_np = compute_graph_properties_numpy(state_rand, n_stress, triu_r15, triu_c15)
        if result_np is None:
            stress_fail += 1
            continue

        mu_np, dv_np, mv_np, A_np = result_np
        if (abs(mu_np - mu_nx) < 1e-10 and
            np.allclose(dv_np, dv_nx, atol=1e-10) and
            np.allclose(mv_np, mv_nx, atol=1e-10)):
            stress_pass += 1
        else:
            stress_fail += 1
            print(f"    Test {t}: MISMATCH mu_diff={abs(mu_np-mu_nx):.2e}")

    stress_ok = stress_fail == 0
    if not stress_ok:
        numpy_pass = False
    print(f"    Results: {stress_pass}/{n_tests} passed, {stress_fail} failed -> "
          f"[{'PASS' if stress_ok else 'FAIL'}]")

    print("=" * 65)
    if numpy_pass:
        print("All numpy cross-verification tests PASSED.")
    else:
        print("WARNING: Some numpy tests FAILED!")

    return all_pass and numpy_pass

unit_tests_passed = run_unit_tests()

In [ ]:
"""Cell 5.5: Performance Comparison — NetworkX vs Numpy Bypass"""

def benchmark_performance(n=15, n_trials=200):
    """Time NetworkX vs numpy reward computation.

    Args:
        n: number of vertices
        n_trials: number of reward computations to time
    """
    decisions = n * (n - 1) // 2
    triu_r, triu_c = precompute_triu_indices(n)

    # Generate random connected graph states
    rng = np.random.RandomState(123)
    states = []
    while len(states) < n_trials:
        s = rng.randint(0, 2, size=decisions).astype(np.int32)
        # Only use connected graphs for fair comparison
        G = build_graph(s, n)
        if nx.is_connected(G):
            states.append(s)

    print(f"Performance benchmark: n={n}, {n_trials} connected graphs")
    print("-" * 55)

    # ── NetworkX timing ──
    t0 = time.perf_counter()
    nx_results = []
    for s in states:
        G = build_graph(s, n)
        mu = compute_laplacian_spectral_radius(G)
        dv, mv = compute_dv_mv(G, n)
        valid = dv > 0
        bound_vals = 2.0 * mv[valid]**2 / dv[valid]
        nx_results.append(mu - np.max(bound_vals))
    t_nx = time.perf_counter() - t0

    # ── Numpy timing ──
    t0 = time.perf_counter()
    np_results = []
    for s in states:
        result = compute_graph_properties_numpy(s, n, triu_r, triu_c)
        if result is None:
            np_results.append(DISCONNECTED_PENALTY)
        else:
            mu, dv, mv, A = result
            valid = dv > 0
            bound_vals = 2.0 * mv[valid]**2 / dv[valid]
            np_results.append(mu - np.max(bound_vals))
    t_np = time.perf_counter() - t0

    speedup = t_nx / t_np if t_np > 0 else float('inf')

    print(f"  NetworkX: {t_nx:.3f}s ({t_nx/n_trials*1000:.2f} ms/graph)")
    print(f"  Numpy:    {t_np:.3f}s ({t_np/n_trials*1000:.2f} ms/graph)")
    print(f"  Speedup:  {speedup:.1f}x")

    # Verify results match
    max_diff = max(abs(a - b) for a, b in zip(nx_results, np_results))
    print(f"  Max result difference: {max_diff:.2e}")
    print(f"  Results match: {'YES' if max_diff < 1e-10 else 'NO'}")
    print("-" * 55)

    return speedup

perf_speedup = benchmark_performance(n=15, n_trials=200)

In [ ]:
"""Cell 6: Reward Functions — Laplacian Spectral Radius Bounds"""

from functools import partial

# ── NetworkX versions (retained for verify_counterexample) ───────

def calc_score_bound2_nx(state, n):
    """Benchmark bound (DISPROVED): mu <= max_v (2 * m_v^2 / d_v) [NetworkX version]"""
    G = build_graph(state, n)
    if not nx.is_connected(G):
        return DISCONNECTED_PENALTY
    mu = compute_laplacian_spectral_radius(G)
    dv, mv = compute_dv_mv(G, n)
    bound_values = []
    for v in range(n):
        if dv[v] > 0:
            bound_values.append(2.0 * mv[v]**2 / dv[v])
    if len(bound_values) == 0:
        return DISCONNECTED_PENALTY
    return mu - max(bound_values)


def calc_score_bound1_nx(state, n):
    """Attack bound (OPEN): mu <= max_v sqrt(4 * d_v^3 / m_v) [NetworkX version]"""
    G = build_graph(state, n)
    if not nx.is_connected(G):
        return DISCONNECTED_PENALTY
    mu = compute_laplacian_spectral_radius(G)
    dv, mv = compute_dv_mv(G, n)
    bound_values = []
    for v in range(n):
        if dv[v] > 0 and mv[v] > 0:
            bound_values.append(np.sqrt(4.0 * dv[v]**3 / mv[v]))
    if len(bound_values) == 0:
        return DISCONNECTED_PENALTY
    return mu - max(bound_values)


def calc_score_bound4_nx(state, n):
    """Secondary attack (OPEN): mu <= max_v (2 * d_v^2 / m_v) [NetworkX version]"""
    G = build_graph(state, n)
    if not nx.is_connected(G):
        return DISCONNECTED_PENALTY
    mu = compute_laplacian_spectral_radius(G)
    dv, mv = compute_dv_mv(G, n)
    bound_values = []
    for v in range(n):
        if dv[v] > 0 and mv[v] > 0:
            bound_values.append(2.0 * dv[v]**2 / mv[v])
    if len(bound_values) == 0:
        return DISCONNECTED_PENALTY
    return mu - max(bound_values)


def calc_score_bound9_nx(state, n):
    """Secondary attack (OPEN): mu <= max_v (m_v + 3*d_v) / 2 [NetworkX version]"""
    G = build_graph(state, n)
    if not nx.is_connected(G):
        return DISCONNECTED_PENALTY
    mu = compute_laplacian_spectral_radius(G)
    dv, mv = compute_dv_mv(G, n)
    bound_values = []
    for v in range(n):
        if dv[v] > 0:
            bound_values.append((mv[v] + 3.0 * dv[v]) / 2.0)
    if len(bound_values) == 0:
        return DISCONNECTED_PENALTY
    return mu - max(bound_values)


def calc_score_bound33_nx(state, n):
    """Secondary attack (OPEN, edge-max): mu <= max_{vi~vj} (2*(d_i+d_j)-(m_i+m_j)) [NetworkX]"""
    G = build_graph(state, n)
    if not nx.is_connected(G):
        return DISCONNECTED_PENALTY
    mu = compute_laplacian_spectral_radius(G)
    dv, mv = compute_dv_mv(G, n)
    bound_values = []
    for (i, j) in G.edges():
        val = 2.0 * (dv[i] + dv[j]) - (mv[i] + mv[j])
        bound_values.append(val)
    if len(bound_values) == 0:
        return DISCONNECTED_PENALTY
    return mu - max(bound_values)


# ── Numpy bypass versions (used in CEM hot loop) ────────────────

def calc_score_bound2_numpy(state, n, triu_rows, triu_cols):
    """Benchmark bound (DISPROVED): mu <= max_v (2 * m_v^2 / d_v) [numpy version]"""
    result = compute_graph_properties_numpy(state, n, triu_rows, triu_cols)
    if result is None:
        return DISCONNECTED_PENALTY
    mu, dv, mv, A = result
    valid = dv > 0
    if not np.any(valid):
        return DISCONNECTED_PENALTY
    bound_values = np.where(valid, 2.0 * mv**2 / np.maximum(dv, 1e-300), 0.0)
    return mu - np.max(bound_values[valid])


def calc_score_bound1_numpy(state, n, triu_rows, triu_cols):
    """Attack bound (OPEN): mu <= max_v sqrt(4 * d_v^3 / m_v) [numpy version]"""
    result = compute_graph_properties_numpy(state, n, triu_rows, triu_cols)
    if result is None:
        return DISCONNECTED_PENALTY
    mu, dv, mv, A = result
    valid = (dv > 0) & (mv > 0)
    if not np.any(valid):
        return DISCONNECTED_PENALTY
    bound_values = np.where(valid, np.sqrt(4.0 * dv**3 / np.maximum(mv, 1e-300)), 0.0)
    return mu - np.max(bound_values[valid])


def calc_score_bound4_numpy(state, n, triu_rows, triu_cols):
    """Secondary attack (OPEN): mu <= max_v (2 * d_v^2 / m_v) [numpy version]"""
    result = compute_graph_properties_numpy(state, n, triu_rows, triu_cols)
    if result is None:
        return DISCONNECTED_PENALTY
    mu, dv, mv, A = result
    valid = (dv > 0) & (mv > 0)
    if not np.any(valid):
        return DISCONNECTED_PENALTY
    bound_values = np.where(valid, 2.0 * dv**2 / np.maximum(mv, 1e-300), 0.0)
    return mu - np.max(bound_values[valid])


def calc_score_bound9_numpy(state, n, triu_rows, triu_cols):
    """Secondary attack (OPEN): mu <= max_v (m_v + 3*d_v) / 2 [numpy version]"""
    result = compute_graph_properties_numpy(state, n, triu_rows, triu_cols)
    if result is None:
        return DISCONNECTED_PENALTY
    mu, dv, mv, A = result
    valid = dv > 0
    if not np.any(valid):
        return DISCONNECTED_PENALTY
    bound_values = np.where(valid, (mv + 3.0 * dv) / 2.0, 0.0)
    return mu - np.max(bound_values[valid])


def calc_score_bound33_numpy(state, n, triu_rows, triu_cols):
    """Secondary attack (OPEN, edge-max): mu <= max_{vi~vj} (2*(d_i+d_j)-(m_i+m_j)) [numpy]"""
    result = compute_graph_properties_numpy(state, n, triu_rows, triu_cols)
    if result is None:
        return DISCONNECTED_PENALTY
    mu, dv, mv, A = result
    edge_rows, edge_cols = np.nonzero(np.triu(A))
    if len(edge_rows) == 0:
        return DISCONNECTED_PENALTY
    d_i, d_j = dv[edge_rows], dv[edge_cols]
    m_i, m_j = mv[edge_rows], mv[edge_cols]
    bound_values = 2.0 * (d_i + d_j) - (m_i + m_j)
    return mu - np.max(bound_values)


# ── Reward function registry ────────────────────────────────────
# fn: NetworkX version (for verify_counterexample and backward compat)
# fn_numpy: numpy bypass version (for CEM hot loop via functools.partial)

BOUND_REGISTRY = {
    'bound1': {'fn': calc_score_bound1_nx, 'fn_numpy': calc_score_bound1_numpy,
               'name': 'Bound 1 (OPEN)',
               'formula': 'mu <= max_v sqrt(4*d_v^3/m_v)'},
    'bound2': {'fn': calc_score_bound2_nx, 'fn_numpy': calc_score_bound2_numpy,
               'name': 'Bound 2 (DISPROVED)',
               'formula': 'mu <= max_v (2*m_v^2/d_v)'},
    'bound4': {'fn': calc_score_bound4_nx, 'fn_numpy': calc_score_bound4_numpy,
               'name': 'Bound 4 (OPEN)',
               'formula': 'mu <= max_v (2*d_v^2/m_v)'},
    'bound9': {'fn': calc_score_bound9_nx, 'fn_numpy': calc_score_bound9_numpy,
               'name': 'Bound 9 (OPEN)',
               'formula': 'mu <= max_v (m_v+3*d_v)/2'},
    'bound33': {'fn': calc_score_bound33_nx, 'fn_numpy': calc_score_bound33_numpy,
                'name': 'Bound 33 (OPEN, edge-max)',
                'formula': 'mu <= max_{vi~vj} (2*(d_i+d_j)-(m_i+m_j))'},
}

# Quick test: compare NetworkX vs numpy on K_5
print("Reward function test on K_5 (n=5, complete graph):")
state_k5 = np.ones(10, dtype=np.int32)
triu_r5, triu_c5 = precompute_triu_indices(5)
for key, info in BOUND_REGISTRY.items():
    score_nx = info['fn'](state_k5, 5)
    score_np = info['fn_numpy'](state_k5, 5, triu_r5, triu_c5)
    match = abs(score_nx - score_np) < 1e-10
    tag = "MATCH" if match else "MISMATCH"
    print(f"  {info['name']}: nx={score_nx:.6f}, np={score_np:.6f} -> [{tag}]")
print("\n(Negative rewards on K_5 are expected — regular graphs satisfy most bounds.)")

In [ ]:
"""Cell 7: CEM Training Engine"""

def generate_sessions(model, n_sessions, n, calc_score_fn):
    """Generate n_sessions graph construction episodes in parallel.

    At each of DECISIONS steps:
      1. Model predicts edge probability for all sessions
      2. Sample binary action (add edge or skip)
      3. Update state vectors

    After all decisions, compute reward for each complete graph.

    Args:
        model: CEMModel instance
        n_sessions: number of parallel sessions
        n: number of vertices
        calc_score_fn: reward function (state, n) -> float

    Returns:
        states_record: numpy array (n_sessions, DECISIONS, 2*DECISIONS)
                       — state at each step for each session
        actions: numpy array (n_sessions, DECISIONS) — action at each step
        rewards: numpy array (n_sessions,) — final reward per session
    """
    decisions = n * (n - 1) // 2
    obs_dim = 2 * decisions

    # Initialize states: all zeros
    current_states = np.zeros((n_sessions, obs_dim), dtype=np.float32)
    current_states[:, decisions] = 1.0

    # Storage for training data
    states_record = np.zeros((n_sessions, decisions, obs_dim), dtype=np.float32)
    actions = np.zeros((n_sessions, decisions), dtype=np.float32)

    model.eval()
    with torch.no_grad():
        for step in range(decisions):
            states_record[:, step, :] = current_states

            state_tensor = torch.from_numpy(current_states).to(DEVICE)
            probs = model(state_tensor).cpu().numpy().flatten()

            random_vals = np.random.random(n_sessions).astype(np.float32)
            action = (random_vals < probs).astype(np.float32)
            actions[:, step] = action

            current_states[action == 1, step] = 1.0

            current_states[:, decisions + step] = 0.0
            if step + 1 < decisions:
                current_states[:, decisions + step + 1] = 1.0

    # Compute rewards for all complete graphs
    rewards = np.zeros(n_sessions, dtype=np.float64)
    for i in range(n_sessions):
        adj_state = current_states[i, :decisions].astype(np.int32)
        rewards[i] = calc_score_fn(adj_state, n)

    return states_record, actions, rewards, current_states[:, :decisions].astype(np.int32).copy()


def select_elites(states_record, actions, rewards, percentile):
    """Select elite sessions and flatten into (state, action) training pairs."""
    threshold = np.percentile(rewards, percentile)
    elite_mask = rewards >= threshold

    elite_states = states_record[elite_mask]
    elite_actions = actions[elite_mask]

    n_elite = elite_states.shape[0]
    decisions = elite_states.shape[1]
    obs_dim = elite_states.shape[2]

    elite_states_flat = elite_states.reshape(-1, obs_dim)
    elite_actions_flat = elite_actions.reshape(-1)

    return elite_states_flat, elite_actions_flat, threshold


def select_super_sessions(states_record, actions, rewards, super_percentile, n_sessions):
    """Select super-elite sessions to carry forward to next generation."""
    threshold = np.percentile(rewards, super_percentile)
    super_mask = rewards >= threshold

    super_states = states_record[super_mask]
    super_actions = actions[super_mask]
    super_rewards = rewards[super_mask]

    sort_idx = np.argsort(-super_rewards)[:n_sessions]

    return super_states[sort_idx], super_actions[sort_idx], super_rewards[sort_idx]


def verify_counterexample(state, n, bound_key):
    """Independently verify a potential counterexample.

    NOTE: This intentionally uses NetworkX (not numpy bypass) for independent
    cross-verification. The CEM loop uses numpy for speed, but verification
    must use a separate code path to catch implementation bugs.
    """
    info = BOUND_REGISTRY[bound_key]
    G = build_graph(state, n)

    if not nx.is_connected(G):
        return False, "Graph is disconnected"

    mu = compute_laplacian_spectral_radius(G)
    dv, mv = compute_dv_mv(G, n)

    # Use NetworkX version for verification
    reward = info['fn'](state, n)

    result = {
        'mu': mu,
        'reward': reward,
        'n_vertices': n,
        'n_edges': G.number_of_edges(),
        'degrees': dv.tolist(),
        'avg_neighbor_degrees': mv.tolist(),
        'is_counterexample': reward > 0,
        'graph': G,
    }

    return reward > 0, result


def train_cem(n, bound_key, max_iter=1000, n_sessions=1000,
              percentile=93, super_percentile=94, lr=0.0001,
              log_interval=20, early_stop_reward=0.0,
              time_limit_seconds=None):
    """Main CEM training loop.

    Uses numpy bypass for reward computation (via functools.partial to bind
    precomputed triu indices). Verification uses NetworkX independently.

    Args:
        n: number of vertices
        bound_key: key into BOUND_REGISTRY
        max_iter: maximum training iterations
        n_sessions: sessions per generation
        percentile: elite selection threshold
        super_percentile: super-elite carry-forward threshold
        lr: SGD learning rate
        log_interval: print every N iterations
        early_stop_reward: stop if best reward exceeds this
        time_limit_seconds: optional time limit

    Returns:
        best_state: adjacency vector of best graph found
        best_reward: reward of best graph
        history: dict with training history
    """
    decisions = n * (n - 1) // 2
    obs_dim = 2 * decisions
    info = BOUND_REGISTRY[bound_key]

    # ── Numpy bypass: precompute triu indices and bind via partial ──
    triu_rows, triu_cols = precompute_triu_indices(n)
    calc_score_fn = partial(info['fn_numpy'], triu_rows=triu_rows, triu_cols=triu_cols)

    print(f"\n{'='*70}")
    print(f"CEM Training: {info['name']}")
    print(f"Formula: {info['formula']}")
    print(f"N={n}, decisions={decisions}, state_dim={obs_dim}")
    print(f"Sessions={n_sessions}, elite={percentile}%, super={super_percentile}%")
    print(f"Max iterations={max_iter}, LR={lr}")
    print(f"Reward computation: numpy bypass (triu indices precomputed)")
    print(f"{'='*70}\n")

    # Create model and optimizer
    model = create_model(n)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    # Initialize super-session storage (empty)
    super_states = np.empty((0, decisions, obs_dim), dtype=np.float32)
    super_actions = np.empty((0, decisions), dtype=np.float32)
    super_rewards = np.empty((0,), dtype=np.float64)
    super_final_adj = np.empty((0, decisions), dtype=np.int32)

    # Tracking
    best_reward = -float('inf')
    best_state = None
    history = {
        'iteration': [],
        'best_reward': [],
        'mean_elite_reward': [],
        'mean_reward': [],
        'elite_threshold': [],
        'time_elapsed': [],
    }

    start_time = time.time()
    counterexample_found = False

    for iteration in range(1, max_iter + 1):
        # Check time limit
        elapsed = time.time() - start_time
        if time_limit_seconds and elapsed > time_limit_seconds:
            print(f"\nTime limit reached ({elapsed:.0f}s / {time_limit_seconds}s). Stopping.")
            break

        # 1. Generate sessions (uses numpy bypass calc_score_fn)
        sess_states, sess_actions, sess_rewards, sess_final_adj = generate_sessions(
            model, n_sessions, n, calc_score_fn
        )

        # 2. Combine with super-sessions from previous generation
        if len(super_rewards) > 0:
            all_states = np.concatenate([sess_states, super_states], axis=0)
            all_actions = np.concatenate([sess_actions, super_actions], axis=0)
            all_rewards = np.concatenate([sess_rewards, super_rewards], axis=0)
            all_final_adj = np.concatenate([sess_final_adj, super_final_adj], axis=0)
        else:
            all_states = sess_states
            all_actions = sess_actions
            all_rewards = sess_rewards
            all_final_adj = sess_final_adj

        # 3. Select elites for training
        elite_states_flat, elite_actions_flat, elite_threshold = select_elites(
            all_states, all_actions, all_rewards, percentile
        )

        # 4. Select super-sessions to carry forward
        super_states, super_actions, super_rewards = select_super_sessions(
            all_states, all_actions, all_rewards, super_percentile, n_sessions
        )
        super_threshold = np.percentile(all_rewards, super_percentile)
        super_mask_adj = all_rewards >= super_threshold
        super_final_adj_candidates = all_final_adj[super_mask_adj]
        super_sort_idx = np.argsort(-all_rewards[super_mask_adj])[:n_sessions]
        super_final_adj = super_final_adj_candidates[super_sort_idx]

        # 5. Train model on elite data
        model.train()
        elite_s_tensor = torch.from_numpy(elite_states_flat).to(DEVICE)
        elite_a_tensor = torch.from_numpy(elite_actions_flat).unsqueeze(1).to(DEVICE)

        batch_size = 2048
        n_samples = elite_s_tensor.shape[0]
        indices = torch.randperm(n_samples)
        total_loss = 0.0
        n_batches = 0

        for start_idx in range(0, n_samples, batch_size):
            end_idx = min(start_idx + batch_size, n_samples)
            batch_idx = indices[start_idx:end_idx]

            batch_states = elite_s_tensor[batch_idx]
            batch_actions = elite_a_tensor[batch_idx]

            optimizer.zero_grad()
            predictions = model(batch_states)
            loss = loss_fn(predictions, batch_actions)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)

        # 6. Track best result
        iter_best_idx = np.argmax(all_rewards)
        iter_best_reward = all_rewards[iter_best_idx]

        if iter_best_reward > best_reward:
            best_reward = iter_best_reward
            best_state = all_final_adj[iter_best_idx].copy()

        # Record history
        elite_mask = all_rewards >= elite_threshold
        mean_elite = np.mean(all_rewards[elite_mask])

        history['iteration'].append(iteration)
        history['best_reward'].append(best_reward)
        history['mean_elite_reward'].append(mean_elite)
        history['mean_reward'].append(np.mean(sess_rewards))
        history['elite_threshold'].append(elite_threshold)
        history['time_elapsed'].append(elapsed)

        # 7. Log progress
        if iteration % log_interval == 0 or iteration == 1:
            print(f"Iter {iteration:5d} | best={best_reward:+.6f} | "
                  f"elite_mean={mean_elite:+.6f} | loss={avg_loss:.4f} | "
                  f"threshold={elite_threshold:+.4f} | time={elapsed:.1f}s")

        # 8. Check for counterexample
        if best_reward > early_stop_reward:
            elapsed = time.time() - start_time
            print(f"\n*** COUNTEREXAMPLE FOUND at iteration {iteration}! ***")
            print(f"    Reward = {best_reward:.6f}")
            print(f"    Time = {elapsed:.1f}s")

            # Verify using NetworkX (independent cross-verification)
            is_valid, details = verify_counterexample(best_state, n, bound_key)
            if is_valid:
                print(f"    VERIFIED: mu={details['mu']:.6f}, reward={details['reward']:.6f}")
                print(f"    Vertices={details['n_vertices']}, Edges={details['n_edges']}")
                print(f"    Degrees: {details['degrees']}")
                counterexample_found = True
            else:
                print(f"    WARNING: Verification failed! {details}")
            break

    elapsed = time.time() - start_time
    print(f"\nTraining complete. Total time: {elapsed:.1f}s")
    print(f"Best reward: {best_reward:+.6f}")

    if not counterexample_found and best_reward > -100:
        print(f"\nNear-miss analysis (best graph, reward={best_reward:.6f}):")
        if best_state is not None:
            G_best = build_graph(best_state, n)
            if nx.is_connected(G_best):
                mu = compute_laplacian_spectral_radius(G_best)
                dv, mv = compute_dv_mv(G_best, n)
                print(f"  mu (spectral radius) = {mu:.6f}")
                print(f"  Edges = {G_best.number_of_edges()}")
                print(f"  Max degree = {max(dv):.0f}, Min degree = {min(dv):.0f}")
                print(f"  Degree sequence: {sorted([int(d) for d in dv], reverse=True)}")

    return best_state, best_reward, history, counterexample_found


print("CEM training engine defined (numpy bypass integrated).")

## Benchmark: Bound 2 (Disproved)

**Formula:** $\mu \leq \max_{v \in V} \frac{2 m_v^2}{d_v}$

This bound was disproved by the SQ* graph at $n=12$. We use it as a benchmark to verify our
CEM implementation works correctly. Finding a counterexample at $n=15$ and $n=20$ should be
relatively straightforward.

In [ ]:
"""Cell 8: Benchmark Run — Bound 2, n=15"""

# Store all results for summary
ALL_RESULTS = {}

print("=" * 70)
print("BENCHMARK: Bound 2, n=15")
print("Expected: Counterexample should be found (bound disproved at n=12)")
print("=" * 70)

bench15_state, bench15_reward, bench15_history, bench15_found = train_cem(
    n=15,
    bound_key='bound2',
    max_iter=500,
    n_sessions=1000,
    percentile=93,
    super_percentile=94,
    lr=0.0001,
    log_interval=20,
    early_stop_reward=0.0,
    time_limit_seconds=3600,  # 1 hour max
)

ALL_RESULTS['bound2_n15'] = {
    'state': bench15_state,
    'reward': bench15_reward,
    'history': bench15_history,
    'found': bench15_found,
    'n': 15,
    'bound': 'bound2',
}

if bench15_found:
    print("\nBenchmark n=15: SUCCESS — Counterexample verified!")
else:
    print(f"\nBenchmark n=15: No counterexample found (best reward: {bench15_reward:.6f})")

In [ ]:
"""Cell 9: Benchmark Run — Bound 2, n=20"""

print("=" * 70)
print("BENCHMARK: Bound 2, n=20")
print("Expected: Counterexample should be found (larger search space)")
print("=" * 70)

bench20_state, bench20_reward, bench20_history, bench20_found = train_cem(
    n=20,
    bound_key='bound2',
    max_iter=1000,
    n_sessions=1000,
    percentile=93,
    super_percentile=94,
    lr=0.0001,
    log_interval=20,
    early_stop_reward=0.0,
    time_limit_seconds=7200,  # 2 hours max
)

ALL_RESULTS['bound2_n20'] = {
    'state': bench20_state,
    'reward': bench20_reward,
    'history': bench20_history,
    'found': bench20_found,
    'n': 20,
    'bound': 'bound2',
}

if bench20_found:
    print("\nBenchmark n=20: SUCCESS — Counterexample verified!")
    # Cross-check: compare with Ghebleh's results
    G_best = build_graph(bench20_state, 20)
    print(f"  Graph: {G_best.number_of_edges()} edges")
    print(f"  Degree sequence: {sorted([d for _, d in G_best.degree()], reverse=True)}")
else:
    print(f"\nBenchmark n=20: No counterexample found (best reward: {bench20_reward:.6f})")

## Attack: Bound 1 (Open Problem)

**Formula:** $\mu \leq \max_{v \in V} \sqrt{\frac{4 d_v^3}{m_v}}$

This bound remains **undisproved**. Ghebleh (2024) tested up to $n=24$ with batch size 200 without
finding a counterexample. We use batch size 1000 (5x more exploration per generation) which may
discover counterexamples they missed.

In [ ]:
"""Cell 10: Attack Run — Bound 1, n=20"""

print("=" * 70)
print("ATTACK: Bound 1 (OPEN), n=20")
print("Target: Find counterexample to unsolved conjecture")
print("=" * 70)

# Estimate time based on benchmark performance
bench_time_per_iter = None
if 'bound2_n20' in ALL_RESULTS and len(ALL_RESULTS['bound2_n20']['history']['time_elapsed']) > 1:
    h = ALL_RESULTS['bound2_n20']['history']
    bench_time_per_iter = h['time_elapsed'][-1] / len(h['iteration'])
    print(f"Estimated time per iteration (from benchmark): {bench_time_per_iter:.1f}s")
    estimated_total = bench_time_per_iter * 1500
    print(f"Estimated total for 1500 iterations: {estimated_total/3600:.1f}h")

attack1_state, attack1_reward, attack1_history, attack1_found = train_cem(
    n=20,
    bound_key='bound1',
    max_iter=1500,
    n_sessions=1000,
    percentile=93,
    super_percentile=94,
    lr=0.0001,
    log_interval=20,
    early_stop_reward=0.0,
    time_limit_seconds=14400,  # 4 hours max
)

ALL_RESULTS['bound1_n20'] = {
    'state': attack1_state,
    'reward': attack1_reward,
    'history': attack1_history,
    'found': attack1_found,
    'n': 20,
    'bound': 'bound1',
}

if attack1_found:
    print("\n*** BREAKTHROUGH: Bound 1 counterexample found at n=20! ***")
    is_valid, details = verify_counterexample(attack1_state, 20, 'bound1')
    if is_valid:
        print(f"  VERIFIED: mu={details['mu']:.6f}, reward={details['reward']:.6f}")
        G = details['graph']
        print(f"  Graph: {G.number_of_edges()} edges")
        print(f"  Degree sequence: {sorted([d for _, d in G.degree()], reverse=True)}")
else:
    print(f"\nAttack n=20: No counterexample found (best reward: {attack1_reward:.6f})")
    print("This is expected — the bound may be true, or counterexamples may exist at larger n.")

In [ ]:
"""Cell 11: Attack Run — Bound 1, n=25 (conditional)"""

# Only run if total elapsed time so far is under 4 hours
total_elapsed = sum(
    r['history']['time_elapsed'][-1] if r['history']['time_elapsed'] else 0
    for r in ALL_RESULTS.values()
)

print(f"Total elapsed time so far: {total_elapsed/3600:.2f}h")

if total_elapsed < 4 * 3600:
    print("Time budget OK — proceeding with n=25 attack.")
    print("=" * 70)
    print("ATTACK: Bound 1 (OPEN), n=25")
    print("Larger search space — 300 edge decisions per graph")
    print("=" * 70)
    
    attack1_25_state, attack1_25_reward, attack1_25_history, attack1_25_found = train_cem(
        n=25,
        bound_key='bound1',
        max_iter=1000,
        n_sessions=1000,
        percentile=93,
        super_percentile=94,
        lr=0.0001,
        log_interval=20,
        early_stop_reward=0.0,
        time_limit_seconds=14400,  # 4 hours max for this run
    )
    
    ALL_RESULTS['bound1_n25'] = {
        'state': attack1_25_state,
        'reward': attack1_25_reward,
        'history': attack1_25_history,
        'found': attack1_25_found,
        'n': 25,
        'bound': 'bound1',
    }
    
    if attack1_25_found:
        print("\n*** BREAKTHROUGH: Bound 1 counterexample found at n=25! ***")
        is_valid, details = verify_counterexample(attack1_25_state, 25, 'bound1')
        if is_valid:
            print(f"  VERIFIED: mu={details['mu']:.6f}, reward={details['reward']:.6f}")
    else:
        print(f"\nAttack n=25: No counterexample found (best reward: {attack1_25_reward:.6f})")
else:
    print(f"Skipping n=25 attack — time budget exceeded ({total_elapsed/3600:.1f}h > 4h).")
    print("This run can be attempted in a separate Kaggle session.")

## Secondary Attacks: Bounds 4, 9, 33

Quick exploration runs for other open bounds at $n=20$. These use shorter training (200-500
iterations) to survey which bounds are most vulnerable to CEM search.

In [ ]:
"""Cell 12: Secondary Attacks — Bounds 4, 9, 33 at n=20"""

# Check remaining time budget
total_elapsed = sum(
    r['history']['time_elapsed'][-1] if r['history']['time_elapsed'] else 0
    for r in ALL_RESULTS.values()
)
print(f"Total elapsed time so far: {total_elapsed/3600:.2f}h")

secondary_targets = [
    ('bound4', 'Bound 4 (OPEN)', 300),
    ('bound9', 'Bound 9 (OPEN)', 300),
    ('bound33', 'Bound 33 (OPEN, edge-max)', 200),
]

remaining_budget = max(0, 10 * 3600 - total_elapsed)  # Up to 10h total
time_per_secondary = remaining_budget / max(len(secondary_targets), 1)

for bound_key, bound_name, max_iter in secondary_targets:
    if time_per_secondary < 300:  # Less than 5 minutes remaining
        print(f"\nSkipping {bound_name} — insufficient time budget.")
        continue
    
    print(f"\n{'='*70}")
    print(f"SECONDARY ATTACK: {bound_name}, n=20")
    print(f"{'='*70}")
    
    sec_state, sec_reward, sec_history, sec_found = train_cem(
        n=20,
        bound_key=bound_key,
        max_iter=max_iter,
        n_sessions=1000,
        percentile=93,
        super_percentile=94,
        lr=0.0001,
        log_interval=20,
        early_stop_reward=0.0,
        time_limit_seconds=time_per_secondary,
    )
    
    ALL_RESULTS[f'{bound_key}_n20'] = {
        'state': sec_state,
        'reward': sec_reward,
        'history': sec_history,
        'found': sec_found,
        'n': 20,
        'bound': bound_key,
    }
    
    if sec_found:
        print(f"\n*** {bound_name} counterexample found! ***")
    else:
        print(f"\n{bound_name}: Best reward = {sec_reward:.6f}")

print("\nSecondary attacks complete.")

## Results Summary and Visualization

Summary table of all CEM runs, best graph visualizations, reward curves, and near-miss analysis.

In [ ]:
"""Cell 13: Results Summary + Visualization"""

# ── Summary Table ────────────────────────────────────────────────
print("=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)
print(f"{'Run':<20} {'N':>3} {'Bound':<15} {'Best Reward':>12} {'Found':>6} {'Iters':>6} {'Time':>8}")
print("-" * 80)

for key, result in ALL_RESULTS.items():
    n = result['n']
    bound = result['bound']
    reward = result['reward']
    found = "YES" if result['found'] else "no"
    iters = len(result['history']['iteration']) if result['history']['iteration'] else 0
    elapsed = result['history']['time_elapsed'][-1] if result['history']['time_elapsed'] else 0
    bound_name = BOUND_REGISTRY[bound]['name']
    
    print(f"{key:<20} {n:>3} {bound_name:<15} {reward:>+12.6f} {found:>6} {iters:>6} {elapsed:>7.1f}s")

print("=" * 80)

# Count counterexamples found
n_found = sum(1 for r in ALL_RESULTS.values() if r['found'])
print(f"\nCounterexamples found: {n_found} / {len(ALL_RESULTS)}")

# ── Reward Curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Best reward over iterations
ax1 = axes[0]
for key, result in ALL_RESULTS.items():
    h = result['history']
    if h['iteration']:
        label = f"{key} (n={result['n']})"
        ax1.plot(h['iteration'], h['best_reward'], label=label, alpha=0.8)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Best Reward')
ax1.set_title('Best Reward over Training')
ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Counterexample threshold')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Mean elite reward over iterations
ax2 = axes[1]
for key, result in ALL_RESULTS.items():
    h = result['history']
    if h['iteration']:
        label = f"{key} (n={result['n']})"
        ax2.plot(h['iteration'], h['mean_elite_reward'], label=label, alpha=0.8)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Mean Elite Reward')
ax2.set_title('Mean Elite Reward over Training')
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Counterexample threshold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Best Graph Visualizations ────────────────────────────────────
# Show graphs for counterexamples and best near-misses
graphs_to_show = []
for key, result in ALL_RESULTS.items():
    if result['state'] is not None and result['reward'] > DISCONNECTED_PENALTY:
        G = build_graph(result['state'], result['n'])
        if nx.is_connected(G):
            graphs_to_show.append((key, G, result['reward'], result['found']))

if graphs_to_show:
    n_graphs = min(len(graphs_to_show), 4)
    # Sort: counterexamples first, then by reward descending
    graphs_to_show.sort(key=lambda x: (-int(x[3]), -x[2]))
    graphs_to_show = graphs_to_show[:n_graphs]
    
    fig, axes = plt.subplots(1, n_graphs, figsize=(5 * n_graphs, 5))
    if n_graphs == 1:
        axes = [axes]
    
    for idx, (key, G, reward, found) in enumerate(graphs_to_show):
        ax = axes[idx]
        pos = nx.spring_layout(G, seed=42)
        
        # Color nodes by degree
        degrees = [G.degree(v) for v in G.nodes()]
        nx.draw(G, pos, ax=ax, node_color=degrees, cmap=plt.cm.viridis,
                node_size=200, font_size=8, with_labels=True,
                edge_color='gray', alpha=0.8)
        
        status = "COUNTEREXAMPLE" if found else f"Near-miss"
        ax.set_title(f"{key}\nreward={reward:+.4f}\n{status}", fontsize=10)
    
    plt.tight_layout()
    plt.savefig('best_graphs.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── Near-Miss Analysis ───────────────────────────────────────────
print("\n" + "=" * 70)
print("NEAR-MISS ANALYSIS")
print("=" * 70)

for key, result in ALL_RESULTS.items():
    if result['state'] is not None and not result['found'] and result['reward'] > DISCONNECTED_PENALTY:
        n = result['n']
        state = result['state']
        G = build_graph(state, n)
        
        if not nx.is_connected(G):
            continue
        
        mu = compute_laplacian_spectral_radius(G)
        dv, mv = compute_dv_mv(G, n)
        
        print(f"\n--- {key} (n={n}) ---")
        print(f"  Best reward: {result['reward']:+.6f}")
        print(f"  mu (Laplacian spectral radius): {mu:.6f}")
        print(f"  Edges: {G.number_of_edges()}")
        print(f"  Degree sequence: {sorted([int(d) for d in dv], reverse=True)}")
        print(f"  Max degree: {max(dv):.0f}, Min degree: {min(dv):.0f}")
        
        # Show gap to counterexample
        bound_fn = BOUND_REGISTRY[result['bound']]['fn']
        bound_formula = BOUND_REGISTRY[result['bound']]['formula']
        print(f"  Bound formula: {bound_formula}")
        print(f"  Gap to violation: {abs(result['reward']):.6f}")

print("\n" + "=" * 70)
print("Analysis complete.")
print("=" * 70)